In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation

from shaft_force_sensing.evaluation import (
    tb_to_numpy,
    add_norm,
    array_bais,
    array_medfilt,
)

In [ ]:
set_name = "C1_S_0002"
save_path = Path("../logs/transformer/ft_m0")
test_path = save_path / "Teleop" / set_name

In [ ]:
gt, pred = tb_to_numpy(test_path)

In [ ]:
# Lazy load, copy values from the original bag
ati_R_base = np.array([
    [-0.8590339472159356, 0.49695129257933546, -0.12288242483909785],
    [-0.49342574435880426, -0.8677290880416204, -0.05981023800669598],
    [-0.1363514295288786, 0.009254327106030052, 0.990617305065506]
])

In [ ]:
hex10_R_base = np.load(Path("../data/Teleop")/ set_name / f"{set_name}.npz")['rotations']
ati_R_hex10 = ati_R_base @ np.linalg.inv(hex10_R_base)

In [ ]:
# Convert gt and pred to ati frame
gt = np.einsum("nij,nj->ni", ati_R_hex10, gt)
pred = np.einsum("nij,nj->ni", ati_R_hex10, pred)

In [ ]:
pred = array_medfilt(pred)
pred = array_bais(pred)

In [ ]:
# Create the 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Set camera view: x+ right, y+ into screen, z+ up (more top-down)
ax.view_init(elev=25, azim=-90)

# Origin points (0,0,0) for each vector
origin = np.zeros(3)

# Plot vectors using quiver
ax.quiver(
    *origin,
    pred[:, 0], pred[:, 1], pred[:, 2],  # x, y, z components
    alpha=0.8, linewidth=5, color='#1f77b4'
)
ax.quiver(
    *origin,
    gt[:, 0], gt[:, 1], gt[:, 2],  # x, y, z components
    alpha=0.5, linewidth=5, color='#ff7f0e'
)

# Set labels
ax.set_xlabel('$F_x$')
ax.set_ylabel('$F_y$')
ax.set_zlabel('$F_z$')

# Set plot limits (optional, for better visibility)
ax.set_xlim([min(pred[:, 0]) - 1, max(pred[:, 0]) + 1])
ax.set_ylim([min(pred[:, 1]) - 1, max(pred[:, 1]) + 1])
ax.set_zlim([min(pred[:, 2]) - 1, max(pred[:, 2]) + 1])

# Show the plot
plt.show()

In [ ]:
# Downsample data from 100Hz to 30Hz
# Take every ~3.33 frames (100/30)
downsample_ratio = 100 / 30
indices = np.arange(0, len(pred), downsample_ratio).astype(int)
pred_30hz = pred[indices]
gt_30hz = gt[indices]

# Create the 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Set camera view: x+ right, y+ into screen, z+ up (more top-down)
ax.view_init(elev=25, azim=-90)

# Set labels with larger font size
ax.set_xlabel('$F_x$', fontsize=14, fontweight='bold')
ax.set_ylabel('$F_y$', fontsize=14, fontweight='bold')
ax.set_zlabel('$F_z$', fontsize=14, fontweight='bold')

# Increase tick label size
ax.tick_params(axis='both', which='major', labelsize=12)

# Set plot limits
ax.set_xlim([min(pred[:, 0]) - 1, max(pred[:, 0]) + 1])
ax.set_ylim([min(pred[:, 1]) - 1, max(pred[:, 1]) + 1])
ax.set_zlim([min(pred[:, 2]) - 1, max(pred[:, 2]) + 1])

# Plot origin
origin = np.array([0, 0, 0])

# Initialize an empty list for quiver plot objects
quivers = []

# Update function for animation
def update(frame):
    # Remove the previous quiver if it exists
    for quiver in quivers:
        quiver.remove()
    quivers.clear()

    # Draw pred vector
    quiver_pred = ax.quiver(
        *origin, *pred_30hz[frame],
        color='#1f77b4', alpha=0.8, linewidth=5,
        label='Predicted'
    )
    quivers.append(quiver_pred)
    
    # Draw gt vector
    quiver_gt = ax.quiver(
        *origin, *gt_30hz[frame],
        color='#ff7f0e', alpha=0.5, linewidth=5,
        label='Ground Truth'
    )
    quivers.append(quiver_gt)
    
    # Add legend (only on first frame to avoid duplicates)
    if frame == 0:
        ax.legend(loc='upper right', fontsize=12)
    
    return quivers

# Create animation with 30 fps
ani = FuncAnimation(fig, update, frames=len(pred_30hz), interval=33.33, blit=False)

# Save animation at 30 fps
ani.save(f'../logs/results/{set_name}_force.mp4', writer='ffmpeg', fps=30)
plt.show()